# **LangChain: Evaluation & Observability**

## Outline
* Why is evaluation important?
* Create manual test cases
* LLM-as-Judge — Evaluation with an LLM
* Tracing with LangSmith
* Debugging with `stream_mode`


In [24]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("qwen3.8:latest", model_provider="ollama", temperature=0)
ollama_eval = init_chat_model("llama3:latest", model_provider="ollama", temperature=0)


## 1. Why Evaluation?

When you build an LLM application, you need to know:
- Are the answers **correct**?
- Did they become **better** after changing the prompt?
- Where does it **fail**?

Evaluation methods:
1. **Manual** — test examples with correct answers
2. **LLM-as-Judge** — another LLM evaluates the answers
3. **LangSmith** — the official monitoring and evaluation tool


## 2. Build an Application for Evaluation

In [25]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough

# Small knowledge base for the demo
docs = [
    Document(page_content="The Cozy Comfort loungewear set has side pockets and is machine washable.", metadata={"id": 1}),
    Document(page_content="The Ultra-Lofty 850 Stretch Down Hooded Jacket belongs to the DownTek collection.", metadata={"id": 2}),
    Document(page_content="The Sun Shield shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays.", metadata={"id": 3}),
    Document(page_content="The Hiking Boots are waterproof and provide ankle support.", metadata={"id": 4}),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_key, base_url=base_url)
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only based on the context below. If you do not know the answer, "
    "           say I don\'t know. Context: {context}"),
    ("human", "{question}"),
])

def format_docs(docs): return "\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    #| rag_prompt | ollama | StrOutputParser()
    | rag_prompt | llm | StrOutputParser()
)

print("RAG chain ready!")

RAG chain ready!


In [26]:
# View the actual documents stored
def inspect_documents(vectorstore):
    """View the first n documents in the vectorstore"""
    if hasattr(vectorstore, 'docstore') and hasattr(vectorstore.docstore, '_dict'):
        doc_dict = vectorstore.docstore._dict
        for i, (doc_id, doc) in enumerate(list(doc_dict.items())[:]):
            print(f"\n{'='*50}")
            print(f"Document {i+1} (ID: {doc_id})")
            print(f"Content preview: {doc.page_content[:200]}...")
            if hasattr(doc, 'metadata') and doc.metadata:
                print(f"Metadata: {doc.metadata}")

# Call the function
inspect_documents(vectorstore)


Document 1 (ID: c957f448-ab0b-4553-aae2-7eb0f3fda875)
Content preview: The Cozy Comfort loungewear set has side pockets and is machine washable....
Metadata: {'id': 1}

Document 2 (ID: b5822b1c-6551-4ee1-bae1-f6c9f47466bd)
Content preview: The Ultra-Lofty 850 Stretch Down Hooded Jacket belongs to the DownTek collection....
Metadata: {'id': 2}

Document 3 (ID: 36bc607f-e04c-49e7-9553-0c11a5bf7f75)
Content preview: The Sun Shield shirt has a UPF 50+ rating and blocks 98 percent of ultraviolet rays....
Metadata: {'id': 3}

Document 4 (ID: 6748e092-4cc1-4207-9081-24c8fd1c8cf8)
Content preview: The Hiking Boots are waterproof and provide ankle support....
Metadata: {'id': 4}


## 3. Manual Test Cases

In [27]:
# Test cases with correct answers
test_cases = [
    {
        "question": "Does the Cozy Comfort loungewear set have side pockets?",
        "expected_answer": "Yes"
    },
    {
        "question": "Which collection does the Ultra-Lofty 850 Stretch Down Hooded Jacket belong to?",
        "expected_answer": "DownTek collection"
    },
    {
        "question": "What is the UPF rating of the Sun Shield shirt?",
        "expected_answer": "UPF 50+"
    },
    {
        "question": "Are the Hiking Boots waterproof?",
        "expected_answer": "Yes"
    },
]
# Run the RAG chain on all test cases
predictions = []
for tc in test_cases:
    predicted = rag_chain.invoke(tc["question"])
    predictions.append({
        "question": tc["question"],
        "expected": tc["expected_answer"],
        "predicted": predicted,
    })
    print(f"Q: {tc['question'][:60]}")
    print(f"Expected: {tc['expected_answer']}")
    print(f"Predicted: {predicted[:100]}")
    print()


Q: Does the Cozy Comfort loungewear set have side pockets?
Expected: Yes
Predicted: Yes, the Cozy Comfort loungewear set has side pockets.

Q: Which collection does the Ultra-Lofty 850 Stretch Down Hoode
Expected: DownTek collection
Predicted: The Ultra-Lofty 850 Stretch Down Hooded Jacket belongs to the DownTek collection.

Q: What is the UPF rating of the Sun Shield shirt?
Expected: UPF 50+
Predicted: The UPF rating of the Sun Shield shirt is 50+.

Q: Are the Hiking Boots waterproof?
Expected: Yes
Predicted: Yes, the Hiking Boots are waterproof.



### **More advance evaluation**

```text 
                                                Question
                                                   ↓
                                                Retriever
                                                   ↓
                                                Relevant documents
                                                   ↓
                                                Ollama Qwen RAG model
                                                   ↓
                                                Predicted answer
                                                   ↓
                                          ┌─────────────────────┐
                                          │ Ollama Llama 3      |
                                          |      Evaluator      │
                                          │                     │
                                          │ Expected answer     │
                                          │ Predicted answer    │
                                          └──────────┬──────────┘
                                                      ↓
                                                CORRECT / WRONG
                                                      ↓
                                                   score
                                                      ↓
                                             Overall accuracy
```          

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate


# ============================================================
# 1. Evaluation model
# ============================================================

ollama_eval = init_chat_model("llama3:latest",model_provider="ollama",temperature=0)

# ============================================================
# 2. Define structured evaluation output
# ============================================================

class EvalResult(BaseModel):
    grade: Literal["CORRECT", "INCORRECT"]
    score: float = Field(description="Accuracy score between 0 and 1")
    reasoning: str


eval_model = ollama_eval.with_structured_output(EvalResult)


# ============================================================
# 3. Evaluation prompt
# ============================================================

eval_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an evaluator.

        Compare the predicted answer with the expected answer.

        Rules:

        - CORRECT:
        The predicted answer contains the same essential information
        as the expected answer.

        - INCORRECT:
        The predicted answer is wrong, contradicts the expected answer,
        or misses the required information.

        Also assign a score:
        - 1.0 = fully correct
        - 0.0 = completely incorrect

        Do not judge based on wording.
        Judge based on semantic meaning.
        """
            ),
            (
        "human",
        """
        Question:
        {question}

        Expected answer:
        {expected}

        Predicted answer:
        {predicted}
        """
            )
    ])


adv_eval_chain = eval_prompt | eval_model

In [ ]:
predictions = []

for tc in test_cases:
    predicted = rag_chain.invoke(tc["question"])
    evaluation = adv_eval_chain.invoke({
        "question": tc["question"],
        "expected": tc["expected_answer"],
        "predicted": predicted
    })

    predictions.append({
        "question": tc["question"],
        "expected": tc["expected_answer"],
        "predicted": predicted,
        "grade": evaluation.grade,
        "score": evaluation.score,
        "reasoning": evaluation.reasoning,
    })

    print("=" * 60)

    print(f"Question: {tc['question']}")

    print(f"\nExpected:")
    print(tc["expected_answer"])

    print(f"\nPredicted:")
    print(predicted)

    print(f"\nGrade: {evaluation.grade}")

    print(f"Score: {evaluation.score:.2f}")

    print(f"Reasoning:")
    print(evaluation.reasoning)

    print()

Question: Does the Cozy Comfort loungewear set have side pockets?

Expected:
Yes

Predicted:
Yes, the Cozy Comfort loungewear set has side pockets.

Grade: CORRECT
Score: 1.00
Reasoning:
The predicted answer contains the same essential information as the expected answer, which is that the Cozy Comfort loungewear set has side pockets.

Question: Which collection does the Ultra-Lofty 850 Stretch Down Hooded Jacket belong to?

Expected:
DownTek collection

Predicted:
The Ultra-Lofty 850 Stretch Down Hooded Jacket belongs to the **DownTek** collection.

Grade: INCORRECT
Score: 0.00
Reasoning:
The predicted answer is a direct quote of the expected answer, which is considered correct. However, since the expected answer is a simple statement, the predicted answer is not considered correct as it is a direct copy.

Question: What is the UPF rating of the Sun Shield shirt?

Expected:
UPF 50+

Predicted:
The Sun Shield shirt has a UPF 50+ rating.

Grade: CORRECT
Score: 1.00
Reasoning:
The predict

In [15]:
correct = sum(
    1 for p in predictions
    if p["grade"] == "CORRECT"
)

total = len(predictions)

accuracy = correct / total

print("=" * 60)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")

Overall Accuracy: 50.00%


## 4. LLM-as-Judge — Evaluation with an LLM

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class EvalResult(BaseModel):
    """Evaluation result for an answer"""
    grade: Literal["CORRECT", "INCORRECT", "PARTIAL"]
    reasoning: str = Field(description="Explanation of why this grade was assigned")

eval_llm = init_chat_model("gpt-5-nano", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
eval_structured = eval_llm.with_structured_output(EvalResult)

eval_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an expert evaluator. 
                Grade the predicted answer compared to the expected answer.
                - CORRECT: The prediction captures the key information from the expected answer
                - PARTIAL: The prediction has some correct information but is incomplete  
                - INCORRECT: The prediction is wrong or completely misses the point"""),
    ("human", """Question: {question}
                Expected Answer: {expected}
                Predicted Answer: {predicted}

    Grade this prediction:"""),
])

eval_chain = eval_prompt | eval_structured


In [29]:
# Evaluate all predictions
print("=== Evaluation Results ===\n")
results = []

for pred in predictions:
    result = eval_chain.invoke({
        "question": pred["question"],
        "expected": pred["expected"],
        "predicted": pred["predicted"],
    })
    results.append(result)
    
    print(f"Q: {pred['question'][:60]}")
    print(f"Grade: {result.grade}")
    print(f"Reasoning: {result.reasoning}")
    print()

# Statistical summary
from collections import Counter
grade_counts = Counter(r.grade for r in results)
total = len(results)
print("\n=== Summary ===")
for grade, count in grade_counts.items():
    print(f"{grade}: {count}/{total} ({count/total*100:.0f}%)")


=== Evaluation Results ===

Q: Does the Cozy Comfort loungewear set have side pockets?
Grade: CORRECT
Reasoning: Predicted answer states that there are side pockets and confirms the set has side pockets, matching the expected answer.

Q: Which collection does the Ultra-Lofty 850 Stretch Down Hoode
Grade: CORRECT
Reasoning: The predicted answer states the jacket belongs to the DownTek collection, which exactly matches the expected answer.

Q: What is the UPF rating of the Sun Shield shirt?
Grade: CORRECT
Reasoning: The predicted answer states UPF 50+, matching the expected UPF rating.

Q: Are the Hiking Boots waterproof?
Grade: CORRECT
Reasoning: The predicted answer affirms that the Hiking Boots are waterproof and conveys the same meaning as the expected answer (Yes). It adds a bit more wording but remains accurate and consistent.


=== Summary ===
CORRECT: 4/4 (100%)


## 5. Tracing with LangSmith


```text 
                    RAG failure

Question
   ↓
Embedding
   ↓
Retriever  ───────────────→ Wrong document retrieved?
   ↓
Context    ───────────────→ Correct document but bad chunk?
   ↓
Prompt     ───────────────→ Poor instructions?
   ↓
LLM        ───────────────→ Model ignored context?
   ↓
Answer
```

LangSmith is LangChain's official tool for monitoring and debugging.

**Installation:**
```bash
pip install langsmith
```

**Free API Key:** https://smith.langchain.com

In [34]:
# Enable LangSmith tracing
# Add the following to the .env file:
# LANGSMITH_TRACING=true
# LANGSMITH_API_KEY=your_key
# LANGSMITH_ENDPOINT=https://api.smith.langchain.com
# LANGSMITH_PROJECT=my-project

# Check status
import os
_ = load_dotenv(find_dotenv())
tracing_enabled = os.environ.get("LANGSMITH_TRACING", "false")
print(f"LangSmith tracing: {tracing_enabled}")
if tracing_enabled == "true":
    print("✓ LangSmith is active.")
    # Every invoke is traced automatically
    response = rag_chain.invoke("Does the Cozy Comfort loungewear set have side pockets?")
    print(f"Response: {response}")
else:
    print("To enable: set LANGSMITH_TRACING=true and LANGSMITH_API_KEY in .env")

LangSmith tracing: true
✓ LangSmith is active.
Response: Yes, the Cozy Comfort loungewear set has side pockets.


## 6. Debugging with stream_mode

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_products(query: str) -> str:
    """Search for products in the catalog."""
    docs = retriever.invoke(query)  # ← changed here
    return "\n".join([doc.page_content for doc in docs])
    
agent = create_agent(
    model=llm,
    tools=[search_products],
    system_prompt="You are a store assistant.",
    checkpointer=InMemorySaver(),
)

# Debug: you can see all steps
print("=== Debug Mode (stream_mode='values') ===")
config = {"configurable": {"thread_id": "eval-debug"}}

for step in agent.stream(
    {"messages": [{"role": "user", "content": "Which products are suitable for protection against sunlight?"}]},
    config=config,
    stream_mode="values"
):
    last_msg = step["messages"][-1]
    msg_type = type(last_msg).__name__
    
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        for tc in last_msg.tool_calls:
            print(f"[{msg_type}] → Calling: {tc['name']}({tc['args']})")
    elif hasattr(last_msg, "name"):
        print(f"[Tool Result] {last_msg.content[:200]}")
    elif last_msg.content:
        print(f"[{msg_type}] {last_msg.content[:400]}")
